# MSDS 420 Final Project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.listdir('/content/drive/MyDrive/MSDS420_Final_Project')



Mounted at /content/drive


['2024_pudb_sf_ctf_fhlmc.csv',
 '2024_pudb_sf_ctf_fnma.csv',
 '2024_st_cnty.csv',
 '2024_cbsa_metro.csv',
 'CountyMortgagesPercent-30-89DaysLate-thru-2025-03.csv',
 'CountyMortgagesPercent-90-plusDaysLate-thru-2025-03.csv',
 'chicago_tracts.csv',
 'chicago_county.csv',
 'fact_loans_chicago_fhlmc.csv',
 'fact_loans_chicago_fnma.csv',
 'dim_county_chicago.csv',
 'dim_borrower_chicago.csv',
 'dim_tract_chicago.csv',
 'fact_delinquency_30_89_chicago.csv',
 'fact_delinquency_90_chicago.csv',
 'Chicago_mortgage_3nf.sql',
 'dim_county_chicago_fixed.csv',
 'dim_county_with_ids.csv',
 'dim_tract_chicago_fixed.csv',
 'dim_borrower_chicago_dedup.csv',
 'dim_borrower_with_ids.csv',
 'dim_tract_with_ids.csv',
 'dim_enterprise_with_ids.csv',
 'dim_cbsa_with_ids.csv',
 'fact_loans_fhlmc_final.csv',
 'fact_loans_fnma_final.csv',
 'fact_delinquency_30_89_final.csv',
 'fact_delinquency_90_plus_final.csv',
 'final_project.ipynb',
 'Morthahe_chicago_warehouse_validation.sql',
 'business_questions_mortgage

In [2]:
import pandas as pd

# Show ALL columns
pd.set_option('display.max_columns', None)

# Show full column width
pd.set_option('display.max_colwidth', None)

# Prevent wrapping
pd.set_option('display.expand_frame_repr', False)

# Increase display width
pd.set_option('display.width', 2000)


In [3]:
import pandas as pd

base_path = "/content/drive/MyDrive/MSDS420_Final_Project/"

files = [
    "2024_pudb_sf_ctf_fhlmc.csv",
    "2024_pudb_sf_ctf_fnma (1).csv",
    "2024_pudb_sf_ctf_fnma.csv",
    "2024_st_cnty.csv",
    "2024_cbsa_metro.csv",
    "CountyMortgagesPercent-30-89DaysLate-thru-2025-03.csv",
    "CountyMortgagesPercent-90-plusDaysLate-thru-2025-03.csv"
]

for file in files:
    print(f"\n\n================ {file} ================\n")

    try:
        df = pd.read_csv(base_path + file, low_memory=False)
        print(df.head())
    except Exception as e:
        print(f"Error loading {file}: {e}")




================ 2024_pudb_sf_ctf_fhlmc.csv ================

   enterprise  record_num_sf_ctf  state_fips  cbsa_metro_code  county_fips  tract_2020  tract_minority_pct  tract_income_med  ami_local  tract_income_ratio  income_annual  ami_hud  income_ratio  upb_acq  purpose_ctf  fed_guarantee_ctf  borr_num  fthb  race1_borr  race2_borr  race3_borr  race4_borr  race5_borr  ethnicity_borr  race1_coborr  race2_coborr  race3_coborr  race4_coborr  race5_coborr  ethnicity_coborr  sex_borr  sex_coborr  age_borr_cat  age_coborr_cat  occupancy_sf_ctf  rate_spread  hoepa  property_type  lien_status  age_borr_62_cat  age_coborr_62_cat   ltv  same_year_acq  term_orig  units_num  rate_orig  upb_orig  preapproval  channel_apply  aus  score_borr_model  score_coborr_model  dti_cat  points  period_intro_rate  mh_land_interest  property_value  tract_rural  county_lower_ms_delta  county_mid_appalachia  county_persistent_poverty  area_concentrated_poverty  area_high_opp  tract_colonias
0           2     

# STEP 1 — Filter FNMA and FHLMC Separately

Assuming we have: fhlmc, fnma

In [ ]:
fhlmc = pd.read_csv(base_path + "2024_pudb_sf_ctf_fhlmc.csv", low_memory=False)
fnma = pd.read_csv(base_path + "2024_pudb_sf_ctf_fnma (1).csv", low_memory=False)

fhlmc_chicago = fhlmc[
    (fhlmc["state_fips"] == 17) &
    (fhlmc["cbsa_metro_code"] == 16980)
].copy()

fnma_chicago = fnma[
    (fnma["state_fips"] == 17) &
    (fnma["cbsa_metro_code"] == 16980)
].copy()

print("FHLMC Chicago:", fhlmc_chicago.shape)
print("FNMA Chicago:", fnma_chicago.shape)


# STEP 2 — Create Chicago Loan Fact Tables (Separate Enterprises)
Keep only needed columns from each:

In [ ]:
loan_cols = [
    "record_num_sf_ctf",
    "enterprise",
    "state_fips",
    "county_fips",
    "cbsa_metro_code",
    "tract_2020",
    "income_annual",
    "ltv",
    "dti_cat",
    "rate_orig",
    "upb_orig",
    "property_value",
    "term_orig",
    "purpose_ctf"
]

fhlmc_fact = fhlmc_chicago[loan_cols].copy()
fnma_fact = fnma_chicago[loan_cols].copy()

In [ ]:
fhlmc_fact.to_csv(base_path + "fact_loans_chicago_fhlmc.csv", index=False)
fnma_fact.to_csv(base_path + "fact_loans_chicago_fnma.csv", index=False)

This keeps enterprises separate (clean modeling practice)

# STEP 3 — Chicago Tract Dimension
Create from BOTH but deduplicate:


In [ ]:
tract_cols = [
    "tract_2020",
    "tract_minority_pct",
    "tract_income_med",
    "tract_income_ratio",
    "area_concentrated_poverty",
    "area_high_opp",
    "county_fips"
]

tract_fhlmc = fhlmc_chicago[tract_cols]
tract_fnma = fnma_chicago[tract_cols]

dim_tract_chicago = pd.concat([tract_fhlmc, tract_fnma]).drop_duplicates()

dim_tract_chicago.to_csv(base_path + "dim_tract_chicago.csv", index=False)


# STEP 4 — Chicago County Dimension
Do NOT hardcode Cook.

In [ ]:
county_fhlmc = fhlmc_chicago[["state_fips", "county_fips"]]
county_fnma = fnma_chicago[["state_fips", "county_fips"]]

dim_county_chicago = pd.concat([county_fhlmc, county_fnma]).drop_duplicates()

dim_county_chicago.to_csv(base_path + "dim_county_chicago.csv", index=False)


Later we join with 2024_st_cnty to get names.

# STEP 5 — Borrower Dimension (Chicago Only)

In [ ]:
borrower_cols = [
    "record_num_sf_ctf",
    "race1_borr",
    "ethnicity_borr",
    "sex_borr",
    "age_borr_cat",
    "fthb"
]

borrower_fhlmc = fhlmc_chicago[borrower_cols]
borrower_fnma = fnma_chicago[borrower_cols]

dim_borrower_chicago = pd.concat([borrower_fhlmc, borrower_fnma])

dim_borrower_chicago.to_csv(base_path + "dim_borrower_chicago.csv", index=False)


# STEP 6 — Delinquency (Chicago Counties Only)
From delinquency CSV:

First clean FIPS:

In [ ]:
delinq_30 = pd.read_csv(
    base_path + "CountyMortgagesPercent-30-89DaysLate-thru-2025-03.csv"
)

delinq_90 = pd.read_csv(
    base_path + "CountyMortgagesPercent-90-plusDaysLate-thru-2025-03.csv"
)


In [ ]:
delinq_30["FIPSCode"] = delinq_30["FIPSCode"].str.replace("'", "")
delinq_30["FIPSCode"] = pd.to_numeric(delinq_30["FIPSCode"], errors="coerce")

delinq_90["FIPSCode"] = delinq_90["FIPSCode"].str.replace("'", "")
delinq_90["FIPSCode"] = pd.to_numeric(delinq_90["FIPSCode"], errors="coerce")

Now keep only Chicago counties:

Get Chicago county FIPS list:

In [ ]:
chicago_counties = dim_county_chicago["county_fips"].unique()

chicago_fips_full = [
    int("17" + str(c).zfill(3)) for c in chicago_counties
]

delinq_30_chicago = delinq_30[
    delinq_30["FIPSCode"].isin(chicago_fips_full)
]

delinq_30_chicago.to_csv(base_path + "fact_delinquency_30_89_chicago.csv", index=False)


In [ ]:
chicago_counties = dim_county_chicago["county_fips"].unique()

chicago_fips_full = [
    int("17" + str(c).zfill(3)) for c in chicago_counties
]

delinq_90_chicago = delinq_90[
    delinq_90["FIPSCode"].isin(chicago_fips_full)
]

delinq_90_chicago.to_csv(base_path + "fact_delinquency_90_chicago.csv", index=False)


In [ ]:
print(chicago_fips_full)


[17031, 17043, 17063, 17197, 17097, 17093, 17037, 17089, 17111]


In [ ]:
delinq_30["FIPSCode"].unique()[:10]


array([  nan, 1003., 1073., 1081., 1089., 1097., 1101., 1117., 1125.,
       2020.])

In [ ]:
delinq_90["FIPSCode"].unique()[:10]


array([  nan, 1003., 1073., 1081., 1089., 1097., 1101., 1117., 1125.,
       2020.])

In [ ]:
delinq_30_chicago["FIPSCode"].unique()

array([17031., 17043., 17089., 17093., 17097., 17111., 17197.])

In [ ]:
delinq_90_chicago["FIPSCode"].unique()

array([17031., 17043., 17089., 17093., 17097., 17111., 17197.])

In [ ]:
# Add state_id column (replace 1 if your state_id is different)
dim_county_chicago["state_id"] = 1

# Optional: reorder columns to match table
dim_county_chicago = dim_county_chicago[[
    "state_id",
    "county_fips"
]]

dim_county_chicago.to_csv(base_path + "dim_county_chicago_fixed.csv", index=False)


In [ ]:
dim_county_sql = pd.read_csv(base_path + "dim_county_with_ids.csv")
dim_tract = pd.read_csv(base_path + "dim_tract_chicago.csv")

In [ ]:
dim_tract = dim_tract.merge(
    dim_county_sql,
    on="county_fips",
    how="left"
)

In [ ]:
dim_tract_final = dim_tract[[
    "county_id",
    "tract_2020",
    "tract_minority_pct",
    "tract_income_med",
    "tract_income_ratio",
    "area_concentrated_poverty",
    "area_high_opp"
]]

dim_tract_final.to_csv(base_path + "dim_tract_chicago_fixed.csv", index=False)


In [ ]:
borrower_cols = [
    "race1_borr",
    "ethnicity_borr",
    "sex_borr",
    "age_borr_cat",
    "fthb"
]

borrower_fhlmc = fhlmc_chicago[borrower_cols]
borrower_fnma = fnma_chicago[borrower_cols]

dim_borrower_chicago = pd.concat([borrower_fhlmc, borrower_fnma])

# Now drop duplicates properly
dim_borrower_chicago = dim_borrower_chicago.drop_duplicates()

dim_borrower_chicago.to_csv(base_path + "dim_borrower_chicago_dedup.csv", index=False)


In [ ]:
dim_borrower_sql = pd.read_csv(base_path + "dim_borrower_with_ids.csv")
dim_tract_sql = pd.read_csv(base_path + "dim_tract_with_ids.csv")
dim_enterprise_sql = pd.read_csv(base_path + "dim_enterprise_with_ids.csv")
dim_cbsa_sql = pd.read_csv(base_path + "dim_cbsa_with_ids.csv")


In [ ]:
fact_fhlmc = fhlmc_fact.copy()


In [ ]:
fact_fhlmc = fact_fhlmc.merge(
    dim_enterprise_sql,
    left_on="enterprise",
    right_on="enterprise_code",
    how="left"
)


In [ ]:
print(dim_borrower_sql.columns)
print(fact_fhlmc.columns)


Index(['borrower_id', 'race1_borr', 'ethnicity_borr', 'sex_borr', 'age_borr_cat', 'fthb'], dtype='object')
Index(['record_num_sf_ctf', 'enterprise', 'state_fips', 'county_fips', 'cbsa_metro_code', 'tract_2020', 'income_annual', 'ltv', 'dti_cat', 'rate_orig', 'upb_orig', 'property_value', 'term_orig', 'purpose_ctf', 'enterprise_id', 'enterprise_code'], dtype='object')


In [ ]:
fact_fhlmc = fhlmc_chicago.copy()


In [ ]:
print(fact_fhlmc.columns)


Index(['enterprise', 'record_num_sf_ctf', 'state_fips', 'cbsa_metro_code', 'county_fips', 'tract_2020', 'tract_minority_pct', 'tract_income_med', 'ami_local', 'tract_income_ratio', 'income_annual', 'ami_hud', 'income_ratio', 'upb_acq', 'purpose_ctf', 'fed_guarantee_ctf', 'borr_num', 'fthb', 'race1_borr', 'race2_borr', 'race3_borr', 'race4_borr', 'race5_borr', 'ethnicity_borr', 'race1_coborr', 'race2_coborr', 'race3_coborr', 'race4_coborr', 'race5_coborr', 'ethnicity_coborr', 'sex_borr', 'sex_coborr', 'age_borr_cat', 'age_coborr_cat', 'occupancy_sf_ctf', 'rate_spread', 'hoepa', 'property_type', 'lien_status', 'age_borr_62_cat', 'age_coborr_62_cat', 'ltv', 'same_year_acq', 'term_orig', 'units_num', 'rate_orig', 'upb_orig', 'preapproval', 'channel_apply', 'aus', 'score_borr_model', 'score_coborr_model', 'dti_cat', 'points', 'period_intro_rate', 'mh_land_interest', 'property_value', 'tract_rural', 'county_lower_ms_delta', 'county_mid_appalachia', 'county_persistent_poverty', 'area_concentr

In [ ]:
fact_fhlmc.columns = fact_fhlmc.columns.str.strip()
dim_borrower_sql.columns = dim_borrower_sql.columns.str.strip()
dim_tract_sql.columns = dim_tract_sql.columns.str.strip()
dim_enterprise_sql.columns = dim_enterprise_sql.columns.str.strip()
dim_cbsa_sql.columns = dim_cbsa_sql.columns.str.strip()


In [ ]:
fact_fhlmc = fact_fhlmc.merge(
    dim_enterprise_sql,
    left_on="enterprise",
    right_on="enterprise_code",
    how="left"
)


In [ ]:
fact_fhlmc["enterprise_id"].isnull().sum()


np.int64(0)

In [ ]:
fact_fhlmc = fact_fhlmc.merge(
    dim_borrower_sql,
    on=["race1_borr", "ethnicity_borr", "sex_borr", "age_borr_cat", "fthb"],
    how="left"
)


In [ ]:
fact_fhlmc["borrower_id"].isnull().sum()


np.int64(0)

In [ ]:
fact_fhlmc = fact_fhlmc.merge(
    dim_tract_sql,
    on="tract_2020",
    how="left"
)


In [ ]:
fact_fhlmc["tract_id"].isnull().sum()


np.int64(0)

In [ ]:
fact_fhlmc = fact_fhlmc.merge(
    dim_cbsa_sql,
    on="cbsa_metro_code",
    how="left"
)


In [ ]:
fact_fhlmc["cbsa_id"].isnull().sum()


np.int64(0)

In [ ]:
fact_fhlmc_final = fact_fhlmc[[
    "borrower_id",
    "tract_id",
    "cbsa_id",
    "enterprise_id",
    "income_annual",
    "ltv",
    "dti_cat",
    "rate_orig",
    "upb_orig",
    "property_value",
    "term_orig",
    "purpose_ctf"
]]


In [ ]:
fact_fhlmc_final.to_csv(base_path + "fact_loans_fhlmc_final.csv", index=False)


In [ ]:
fact_fhlmc_final.isnull().sum()


,0
borrower_id,0
tract_id,0
cbsa_id,0
enterprise_id,0
income_annual,0
ltv,0
dti_cat,0
rate_orig,0
upb_orig,0
property_value,0


In [ ]:
print(fhlmc_chicago.shape)
print(fnma_chicago.shape)

(36797, 64)
(29249, 64)


In [ ]:
fact_fnma = fnma_chicago.copy()

print(fact_fnma.shape)



(29249, 64)


In [ ]:
fact_fnma.columns = fact_fnma.columns.str.strip()
dim_borrower_sql.columns = dim_borrower_sql.columns.str.strip()
dim_tract_sql.columns = dim_tract_sql.columns.str.strip()
dim_enterprise_sql.columns = dim_enterprise_sql.columns.str.strip()
dim_cbsa_sql.columns = dim_cbsa_sql.columns.str.strip()


In [ ]:
fact_fnma = fact_fnma.merge(
    dim_enterprise_sql,
    left_on="enterprise",
    right_on="enterprise_code",
    how="left"
)

print("Null enterprise_id:", fact_fnma["enterprise_id"].isnull().sum())


Null enterprise_id: 0


In [ ]:
fact_fnma = fact_fnma.merge(
    dim_borrower_sql,
    on=["race1_borr", "ethnicity_borr", "sex_borr", "age_borr_cat", "fthb"],
    how="left"
)

print("Null borrower_id:", fact_fnma["borrower_id"].isnull().sum())


Null borrower_id: 0


In [ ]:
fact_fnma = fact_fnma.merge(
    dim_tract_sql,
    on="tract_2020",
    how="left"
)

print("Null tract_id:", fact_fnma["tract_id"].isnull().sum())


Null tract_id: 0


In [ ]:
fact_fnma = fact_fnma.merge(
    dim_cbsa_sql,
    on="cbsa_metro_code",
    how="left"
)

print("Null cbsa_id:", fact_fnma["cbsa_id"].isnull().sum())


Null cbsa_id: 0


In [ ]:
fact_fnma_final = fact_fnma[[
    "borrower_id",
    "tract_id",
    "cbsa_id",
    "enterprise_id",
    "income_annual",
    "ltv",
    "dti_cat",
    "rate_orig",
    "upb_orig",
    "property_value",
    "term_orig",
    "purpose_ctf"
]]


In [ ]:
fact_fnma_final.isnull().sum()


,0
borrower_id,0
tract_id,0
cbsa_id,0
enterprise_id,0
income_annual,0
ltv,0
dti_cat,0
rate_orig,0
upb_orig,0
property_value,0


In [ ]:
fact_fnma_final.to_csv(base_path + "fact_loans_fnma_final.csv", index=False)


# STEP 3 — Load Delinquency Data Properly

In [ ]:
delinq_30_chicago = delinq_30_chicago.copy()

delinq_30_long = delinq_30_chicago.melt(
    id_vars=["RegionType", "State", "Name", "FIPSCode"],
    var_name="year_month",
    value_name="delinquency_rate"
)

delinq_30_long["delinquency_type"] = "30_89"

delinq_30_long.head()


,RegionType,State,Name,FIPSCode,year_month,delinquency_rate,delinquency_type
0,County,IL,Cook County,17031.0,2008-01,3.5,30_89
1,County,IL,DuPage County,17043.0,2008-01,1.8,30_89
2,County,IL,Kane County,17089.0,2008-01,2.9,30_89
3,County,IL,Kendall County,17093.0,2008-01,3.0,30_89
4,County,IL,Lake County,17097.0,2008-01,2.6,30_89


# Convert year_month to date

In [ ]:
delinq_30_long["year_month"] = pd.to_datetime(
    delinq_30_long["year_month"],
    format="%Y-%m",
    errors="coerce"
)


# Remove Bad Rows

In [ ]:
delinq_30_long = delinq_30_long.dropna(subset=["year_month"])


# Verify

In [ ]:
delinq_30_long.head()


,RegionType,State,Name,FIPSCode,year_month,delinquency_rate,delinquency_type
0,County,IL,Cook County,17031.0,2008-01-01,3.5,30_89
1,County,IL,DuPage County,17043.0,2008-01-01,1.8,30_89
2,County,IL,Kane County,17089.0,2008-01-01,2.9,30_89
3,County,IL,Kendall County,17093.0,2008-01-01,3.0,30_89
4,County,IL,Lake County,17097.0,2008-01-01,2.6,30_89


In [ ]:
dim_county_sql = pd.read_csv(base_path + "dim_county_with_ids.csv")


In [ ]:
dim_county_sql.head()

,county_id,state_id,county_fips,county_name
0,1,1,31,NaN
1,2,1,43,NaN
2,3,1,63,NaN
3,4,1,197,NaN
4,5,1,97,NaN


# Extract County Portion from FIPSCode

In [ ]:
# Convert to integer first
delinq_30_long["FIPSCode"] = delinq_30_long["FIPSCode"].astype(int)

# Extract last 3 digits (county portion)
delinq_30_long["county_fips"] = delinq_30_long["FIPSCode"] % 1000


In [ ]:
delinq_30_long[["FIPSCode", "county_fips"]].head()


,FIPSCode,county_fips
0,17031,31
1,17043,43
2,17089,89
3,17093,93
4,17097,97


# MERGE COUNTY_ID

In [ ]:
delinq_30_long = delinq_30_long.merge(
    dim_county_sql[["county_id", "county_fips"]],
    on="county_fips",
    how="left"
)


In [ ]:
delinq_30_long["county_id"].isnull().sum()


np.int64(0)

# build the final fact structure for 30-89 delinquency


In [ ]:
delinq_30_final = delinq_30_long[[
    "county_id",
    "year_month",
    "delinquency_rate",
    "delinquency_type"
]]

delinq_30_final.to_csv(base_path + "fact_delinquency_30_89_final.csv", index=False)

delinq_30_final.head()


,county_id,year_month,delinquency_rate,delinquency_type
0,1,2008-01-01,3.5,30_89
1,2,2008-01-01,1.8,30_89
2,8,2008-01-01,2.9,30_89
3,6,2008-01-01,3.0,30_89
4,5,2008-01-01,2.6,30_89


In [ ]:
delinq_30_final.isnull().sum()


,0
county_id,0
year_month,0
delinquency_rate,0
delinquency_type,0


# STEP 4 — 90+ Delinquency

In [ ]:
delinq_90_chicago = delinq_90_chicago.copy()

delinq_90_long = delinq_90_chicago.melt(
    id_vars=["RegionType", "State", "Name", "FIPSCode"],
    var_name="year_month",
    value_name="delinquency_rate"
)

delinq_90_long["delinquency_type"] = "90_plus"


# Convert Date

In [ ]:
delinq_90_long["year_month"] = pd.to_datetime(
    delinq_90_long["year_month"],
    format="%Y-%m",
    errors="coerce"
)

delinq_90_long = delinq_90_long.dropna(subset=["year_month"])


# Extract County Portion

In [ ]:
delinq_90_long["FIPSCode"] = delinq_90_long["FIPSCode"].astype(int)
delinq_90_long["county_fips"] = delinq_90_long["FIPSCode"] % 1000


# Map to Surrogate county_id

In [ ]:
delinq_90_long = delinq_90_long.merge(
    dim_county_sql[["county_id", "county_fips"]],
    on="county_fips",
    how="left"
)

delinq_90_long["county_id"].isnull().sum()


np.int64(0)

#Final Structure



In [ ]:
delinq_90_final = delinq_90_long[[
    "county_id",
    "year_month",
    "delinquency_rate",
    "delinquency_type"
]]

delinq_90_final.to_csv(base_path + "fact_delinquency_90_plus_final.csv", index=False)


In [ ]:
delinq_90_final.head()

,county_id,year_month,delinquency_rate,delinquency_type
0,1,2008-01-01,1.8,90_plus
1,2,2008-01-01,0.8,90_plus
2,8,2008-01-01,1.4,90_plus
3,6,2008-01-01,1.2,90_plus
4,5,2008-01-01,1.0,90_plus
